# Vald

> [Vald](https://github.com/vdaas/vald) 是一个高度可扩展的分布式快速近似最近邻 (ANN) 密集向量搜索引擎。

本笔记本展示了如何使用与 `Vald` 数据库相关的功能。

要运行此笔记本，您需要一个正在运行的 Vald 集群。
有关更多信息，请参阅 [入门](https://github.com/vdaas/vald#get-started)。

请参阅 [安装说明](https://github.com/vdaas/vald-client-python#install)。

In [ ]:
%pip install --upgrade --quiet  vald-client-python langchain-community

## 基本示例

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Vald
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter

raw_documents = TextLoader("state_of_the_union.txt").load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
documents = text_splitter.split_documents(raw_documents)
model_name = "sentence-transformers/all-mpnet-base-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)
db = Vald.from_documents(documents, embeddings, host="localhost", port=8080)

In [ ]:
query = "What did the president say about Ketanji Brown Jackson"
docs = db.similarity_search(query)
docs[0].page_content

### 按向量进行相似性搜索

In [ ]:
embedding_vector = embeddings.embed_query(query)
docs = db.similarity_search_by_vector(embedding_vector)
docs[0].page_content

### 相似度搜索与评分

Similarity search is a technique used to find items that are similar to a given query item. It is often used in applications such as recommendation systems, image retrieval, and natural language processing.

In many similarity search scenarios, it is useful to not only find similar items but also to rank them based on their degree of similarity. This is where the concept of a similarity score comes into play. A similarity score is a numerical value that quantifies how similar two items are. The higher the score, the more similar the items are.

There are various algorithms and metrics that can be used to calculate similarity scores, depending on the type of data and the specific application. Some common approaches include:

*   **Cosine Similarity:** Often used for text data, it measures the cosine of the angle between two non-zero vectors. A score of 1 means the vectors are identical in direction, 0 means they are orthogonal, and -1 means they are diametrically opposed.
*   **Euclidean Distance:** Commonly used for numerical data, it measures the straight-line distance between two points in a multi-dimensional space. In similarity search, a smaller distance indicates higher similarity.
*   **Jaccard Index:** Used for comparing sets, it measures the size of the intersection divided by the size of the union of two sets. It's often used for comparing documents or user behavior.
*   **Dot Product:** A simple measure that can be used when vectors are non-negative. A higher dot product generally indicates greater similarity.

When performing similarity search with scores, the process typically involves:

1.  **Representing Data:** Converting the items (e.g., documents, images, user profiles) into a numerical format, often as vectors.
2.  **Calculating Scores:** Using a chosen similarity metric to compute a score between the query item and all other items in the dataset.
3.  **Ranking Results:** Sorting the items based on their calculated similarity scores in descending order (for similarity) or ascending order (for distance).
4.  **Retrieving Top-K:** Returning the top-k most similar items to the user.

The ability to incorporate similarity scores significantly enhances the utility of similarity search, allowing for more refined and relevant results.

---

相似度搜索是一种用于查找与给定查询项相似的项的技术。它常用于推荐系统、图像检索和自然语言处理等应用。

在许多相似度搜索场景中，不仅找到相似的项而且根据它们的相似程度对它们进行排名是很有用的。这就是相似度评分概念的由来。相似度评分是一个数值，用于量化两个项的相似程度。评分越高，项越相似。

根据数据的类型和具体应用，有多种算法和度量方法可用于计算相似度评分。一些常见的方法包括：

*   **余弦相似度 (Cosine Similarity):** 常用于文本数据，它衡量两个非零向量之间夹角的余弦值。得分为 1 表示向量方向相同，得分为 0 表示它们正交，得分为 -1 表示它们方向相反。
*   **欧氏距离 (Euclidean Distance):** 常用于数值数据，它衡量多维空间中两点之间的直线距离。在相似度搜索中，较小的距离表示较高的相似度。
*   **杰卡德指数 (Jaccard Index):** 用于比较集合，它衡量两个集合交集的大小除以并集的大小。它常用于比较文档或用户行为。
*   **点积 (Dot Product):** 当向量非负时可以使用的一种简单度量。较高的点积通常表示较高的相似度。

在进行带评分的相似度搜索时，该过程通常包括：

1.  **数据表示:** 将项（例如，文档、图像、用户画像）转换为数字格式，通常是向量。
2.  **计算评分:** 使用选定的相似度度量来计算查询项与数据集中所有其他项之间的评分。
3.  **结果排名:** 根据计算出的相似度评分对项进行降序（对于相似度）或升序（对于距离）排序。
4.  **检索 Top-K:** 向用户返回最相似的 K 个项。

结合相似度评分的能力极大地增强了相似度搜索的实用性，从而能够获得更精细和更相关的结果。

In [ ]:
docs_and_scores = db.similarity_search_with_score(query)
docs_and_scores[0]

## 最大化边际相关性搜索 (MMR)

除了在检索器对象中使用相似性搜索，您还可以使用 `mmr` 作为检索器。

In [ ]:
retriever = db.as_retriever(search_type="mmr")
retriever.invoke(query)

或者直接使用 `max_marginal_relevance_search`：

In [ ]:
db.max_marginal_relevance_search(query, k=2, fetch_k=10)

## 安全连接使用示例
为了运行此 Notebook，需要运行一个启用了安全连接的 Vald 集群。

以下是一个使用 [Athenz](https://github.com/AthenZ/athenz) 进行身份验证的 Vald 集群示例，配置如下：

ingress(TLS) -> [authorization-proxy](https://github.com/AthenZ/authorization-proxy)(在 gRPC 元数据中检查 athenz-role-auth) -> vald-lb-gateway

In [ ]:
import grpc

with open("test_root_cacert.crt", "rb") as root:
    credentials = grpc.ssl_channel_credentials(root_certificates=root.read())

# Refresh is required for server use
with open(".ztoken", "rb") as ztoken:
    token = ztoken.read().strip()

metadata = [(b"athenz-role-auth", token)]

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Vald
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter

raw_documents = TextLoader("state_of_the_union.txt").load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
documents = text_splitter.split_documents(raw_documents)
model_name = "sentence-transformers/all-mpnet-base-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

db = Vald.from_documents(
    documents,
    embeddings,
    host="localhost",
    port=443,
    grpc_use_secure=True,
    grpc_credentials=credentials,
    grpc_metadata=metadata,
)

In [ ]:
query = "What did the president say about Ketanji Brown Jackson"
docs = db.similarity_search(query, grpc_metadata=metadata)
docs[0].page_content

### 向量相似性搜索

In [ ]:
embedding_vector = embeddings.embed_query(query)
docs = db.similarity_search_by_vector(embedding_vector, grpc_metadata=metadata)
docs[0].page_content

### 相似度搜索（带分数）

In [ ]:
docs_and_scores = db.similarity_search_with_score(query, grpc_metadata=metadata)
docs_and_scores[0]

### 最大化边际相关性搜索 (MMR)

In [ ]:
retriever = db.as_retriever(
    search_kwargs={"search_type": "mmr", "grpc_metadata": metadata}
)
retriever.invoke(query, grpc_metadata=metadata)

或者：

In [ ]:
db.max_marginal_relevance_search(query, k=2, fetch_k=10, grpc_metadata=metadata)